# A04 — house_price: dữ liệu và so sánh đã lưu

Các framework dùng hai môi trường WSL riêng. Mã huấn luyện ở `training/house_price_keras.ipynb` và `training/house_price_torch.ipynb`. Notebook này đọc kết quả gốc; Run All không train/TEST. Toàn bộ notebook huấn luyện nguyên byte và output cũ nằm trong provenance.


In [ ]:
# Migration r2: select this assignment's original logical paths, preserving frozen source.
from pathlib import Path as _Path
import sys as _sys
_repo = next(p for p in (_Path.cwd(), *_Path.cwd().parents) if (p / 'ASSIGN_REORGANIZATION_EXECUTION_PLAN.md').exists())
_sys.path.insert(0, str(_repo / '_shared'))
from assignment_runtime import activate as _activate
_migration = _activate('A04', 'torch')
_migration


# 1. Assignment Overview

Cell này làm gì?

Notebook này tiếp tục bài toán A03 Housing Price: dùng đặc trưng property để dự đoán giá nhà liên tục. A04 dùng King County House Sales trong `datasets/housing_price/a04`, tập trung vào deep learning regression trên dữ liệu tabular và so sánh cẩn thận với A03.


Output cho thấy điều gì?

Bảng xác nhận A04 giữ đúng bài toán regression của A03: đầu vào là feature mô tả nhà/property, target là giá bán liên tục. Dataset A04 đổi sang King County nên metric cuối chỉ so sánh workflow và chất lượng dữ liệu, không dùng như benchmark trực tiếp với A03.


# 2. Environment Setup

Cell này làm gì?

Import thư viện, cố định seed, kiểm tra TensorFlow/PyTorch version và thiết bị chạy. Notebook dùng `pathlib.Path` để path portable trong repo.


Output cho thấy điều gì?

Environment có đủ hai framework để chạy cùng một dữ liệu đã preprocess. CPU vẫn phù hợp vì King County có khoảng vài chục nghìn dòng, nhỏ hơn nhiều so với các bài toán ảnh/video.


# 3. Load Dataset

Cell này làm gì?

Đọc raw CSV A04 từ `datasets/housing_price/a04`. File gốc được giữ nguyên; mọi xử lý nằm trong DataFrame và pipeline preprocessing.


Output cho thấy điều gì?

Dataset được đọc từ path tương đối trong project. `price` là target regression liên tục; các cột còn lại mô tả thời gian bán, diện tích, vị trí, tình trạng và đặc điểm property.


# 4. Dataset Inspection

Cell này làm gì?

Kiểm tra schema, missing values, duplicates, phân phối target, descriptive statistics và các giá trị housing bất thường.


Output cho thấy điều gì?

Dataset gốc không có missing value và không có full duplicate rows, nhưng có repeated property `id` do một số nhà được bán nhiều lần. Đây là rủi ro leakage nếu cùng một property xuất hiện ở nhiều split, nên bước chuẩn bị sẽ giữ latest sale per `id` trước khi chia train/validation/test. Phân phối `price` lệch phải rõ rệt, vì vậy notebook dùng log1p target cho training và inverse transform khi báo metrics trên price scale gốc.


# 5. Data Preparation

Cell này làm gì?

Tạo feature engineering từ `date`, loại identifier `id`, chia train/validation/test theo price bins, fit preprocessing chỉ trên train, và quyết định target log-transform dựa trên skew.


Output cho thấy điều gì?

Preprocessing bắt đầu bằng cách bỏ full duplicates, loại `price <= 0` nếu có, và giữ latest sale per property `id` để tránh cùng một property đi qua nhiều split. Split được stratify theo price bins; preprocessor chỉ fit trên train set, còn validation/test chỉ dùng `transform`, nên scaling/encoding không học từ held-out data.


# 6. Deep-Learning Data Representation

Cell này làm gì?

In shape của dữ liệu deep learning và đánh giá CNN suitability cho housing-price tabular regression.


Output cho thấy điều gì?

`X_train` là ma trận `(samples, transformed features)`, trong đó zipcode đã được one-hot encode và numeric features đã scale. Đây là vector tabular; MLP regression là lựa chọn tự nhiên hơn CNN.


# 7. Discover Neural Network and CNN Operations

Cell này làm gì?

Minh họa Dense/Linear, ReLU, output regression, Conv1D và pooling bằng toy tensors.


Output cho thấy điều gì?

Regression output là một scalar tuyến tính, không phải xác suất hay class label. Conv1D/pooling chỉ minh họa khi dữ liệu có trục thứ tự; King County sau preprocessing không có cấu trúc cục bộ như vậy.


# 8. Neural Network / CNN From Scratch

Cell này làm gì?

Tự implement bằng NumPy các operation giáo dục: linear layer, ReLU, forward network, output regression, MSE/MAE helper và Conv1D toy.


Output cho thấy điều gì?

NumPy forward pass minh họa cách MLP regression tạo một giá trị liên tục. MSE/MAE được tính trên target đã transform trong toy example; khi đánh giá model thật, notebook chuyển prediction về price scale gốc.


# 9. Deep Learning with Keras/TensorFlow

Cell này làm gì?

Định nghĩa helper metric regression và baseline Keras MLP với output scalar tuyến tính.


Output cho thấy điều gì?

Keras baseline nhận vector property đã transform và xuất một scalar linear trên target đã chuẩn hóa. Loss Huber phù hợp với target có skew/outliers vì bớt nhạy với sai số cực lớn hơn MSE thuần.


# 10. Keras Training

Cell này làm gì?

Train baseline Keras với validation, EarlyStopping và ModelCheckpoint. Thời gian train được đo thật.


Output cho thấy điều gì?

Training log hiển thị loss và MAE trên target đã transform cho train/validation. Checkpoint chọn epoch có validation loss thấp nhất; test set vẫn chưa được dùng.


# 11. Keras Training Curves

Cell này làm gì?

Vẽ loss và MAE của baseline Keras trên train/validation, đồng thời lưu figure.


Output cho thấy điều gì?

Curves cho biết model học ổn định hay overfit trên validation. Vì target đã log/scale, magnitude của loss dùng để chọn model; metric giá nhà dễ hiểu hơn được tính riêng sau khi inverse transform.


# 12. Keras Evaluation

Cell này làm gì?

Đánh giá baseline Keras trên validation set bằng MAE, MSE, RMSE, R²; lưu actual-vs-predicted, residual plot và error distribution.


Output cho thấy điều gì?

Keras baseline validation đạt MAE 70,219, RMSE 138,387, R² 0.8679. Diagnostic plots cho thấy sai số vẫn lớn hơn ở vùng giá cao, phù hợp với target có đuôi phải dài.


# 13. Deep Learning with PyTorch

Cell này làm gì?

Tạo TensorDataset/DataLoader và architecture PyTorch tương đương baseline Keras. Trước train, chạy một batch qua model để kiểm tra shape và parameter count.


Output cho thấy điều gì?

Batch input có dạng `(batch, transformed features)` và output có dạng `(batch, 1)`, đúng với regression scalar. Kiến trúc PyTorch tương đương ý tưởng với Keras baseline để so sánh framework công bằng hơn.


# 14. PyTorch Training

Cell này làm gì?

Train PyTorch baseline bằng loop thủ công, dùng SmoothL1Loss/Huber và lưu checkpoint tốt nhất theo validation loss.


Output cho thấy điều gì?

Loop PyTorch hiển thị đầy đủ `zero_grad`, forward, loss, backward và optimizer step. Validation RMSE/R² trong log đã inverse transform về price scale để dễ đọc hơn loss nội bộ.


# 15. PyTorch Evaluation

Cell này làm gì?

Đánh giá PyTorch baseline trên validation set với cùng metric price-scale như Keras.


Output cho thấy điều gì?

PyTorch baseline được đánh giá trên cùng validation split và cùng target inverse transform. Metric khác biệt giữa hai framework có thể đến từ initialization và training loop, không đủ để kết luận framework nào tốt hơn về bản chất.


# 16. Keras vs PyTorch Comparison

Cell này làm gì?

So sánh baseline MLP Keras và PyTorch bằng validation MAE/RMSE/R².


Output cho thấy điều gì?

Trong baseline framework comparison, PyTorch có validation RMSE thấp nhất (113,519) ở run này. Đây chỉ là kết quả một lần chạy với cùng ý tưởng architecture, không phải kết luận framework nào tốt hơn về bản chất.


# 17. Architecture Experiments

Cell này làm gì?

Train các architecture phù hợp dữ liệu tabular: M1 Basic MLP Regressor, M2 BatchNorm, M3 Residual MLP, M4 Residual MLP + Feature Attention. CNN không được train như production model vì feature order không có ý nghĩa không gian.


Output cho thấy điều gì?

Các architecture đều giữ cùng split, preprocessing, target transform, seed và training budget. CNN được ghi nhận ở mức methodological investigation, không ép thành model vì bài toán là tabular regression.


# 18. Improved Model Comparison

Cell này làm gì?

So sánh M1-M4 theo validation MAE/RMSE/R² và chọn ứng viên tốt nhất bằng validation RMSE thấp nhất.


Output cho thấy điều gì?

`M4 Residual MLP Feature Attention Regressor` là model tốt nhất trên validation với RMSE 119,459, MAE 67,889, R² 0.9015. Sau leakage fix repeated property `id`, M4 attention thắng validation RMSE nhưng vẫn cần cân nhắc vì parameter count cao hơn M1.


# 19. Representation Inspection

Cell này làm gì?

Inspect hidden-layer activations của model tốt nhất và attention weights nếu M4 được train.


Output cho thấy điều gì?

Hidden activations cho thấy model đã biến đổi vector property thành representation phi tuyến trước output giá. Attention weights, nếu dùng M4, là công cụ inspection model chứ không phải bằng chứng nhân quả về thị trường nhà.


# 20. Error Analysis

Cell này làm gì?

Phân tích lỗi validation của model được chọn: absolute error lớn nhất, residual theo price range, và nhóm giá thấp/trung/cao.


Output cho thấy điều gì?

Validation error lớn nhất nằm ở nhóm high-price với MAE khoảng 118,650 và RMSE khoảng 189,305. Điều này phù hợp với phân phối giá lệch phải: nhà giá cao tạo residual lớn hơn và làm RMSE nhạy hơn MAE.


# 21. Final Model

Cell này làm gì?

Load checkpoint tốt nhất theo validation RMSE, evaluate test set đúng một lần, rồi lưu final model và diagnostics.


Output cho thấy điều gì?

Final test evaluation của `M4 Residual MLP Feature Attention Regressor` đạt MAE 70,282, RMSE 146,275, R² 0.8673 trên price scale gốc sau inverse transform. Test set chỉ dùng ở bước này sau khi chọn model bằng validation RMSE.


# 22. A03 vs A04 Comparison

Cell này làm gì?

Đọc artifacts A03 nếu có và so sánh dataset/workflow/metrics với A04. Dataset khác nhau nên không dùng score difference như benchmark trực tiếp.


Output cho thấy điều gì?

A03 và A04 cùng là housing-price regression nhưng khác thị trường, target scale và feature schema. Vì vậy chênh lệch RMSE/R² không chứng minh model A04 tốt hơn hay kém hơn A03; nó chỉ mô tả kết quả trên dataset A04.


# 23. Final Results

Cell này làm gì?

Tổng hợp kết quả cuối, lưu metadata A04 của notebook và liệt kê artifacts đã tạo.


Output cho thấy điều gì?

Notebook đã lưu final model, preprocessor, comparison metrics, error-analysis CSV và figures. Kết quả cuối: selected model là `M4 Residual MLP Feature Attention Regressor`, test MAE 70,282, RMSE 146,275, R² 0.8673; tổng training/evaluation time được ghi nhận khoảng 82.39 giây.


In [ ]:
import pandas as pd
from IPython.display import display, Image
_folder = _repo / 'A04/artifacts/pipeline04/house_price/metrics'
for _file in sorted(_folder.glob('*comparison*.csv')):
    print(_file.name)
    display(pd.read_csv(_file))


In [ ]:
display(Image(filename=str(_repo / 'A04/artifacts/pipeline04/house_price/figures/keras_basic_mlp_training_curves.png')))
